# Coupled Oscillators: Normal Modes, Beating, and Modal Decomposition

**MechanicsDSL Notebook 02** | Classical Mechanics | Intermediate

Two identical pendulums of mass $m$ and length $l$ coupled by a spring of stiffness $k$. This system exhibits normal modes, beating, and complete energy transfer between subsystems.

**MechanicsDSL DSL specification:**
```
\system{coupled_pendulums}
\parameter{m}{1.0}{kg}
\parameter{l}{1.0}{m}
\parameter{k}{0.5}{N/m}
\lagrangian{
    0.5*m*l^2*(\dot{theta1}^2 + \dot{theta2}^2)
    + m*g*l*(cos(theta1) + cos(theta2))
    - 0.5*k*l^2*(theta1 - theta2)^2
}
\target{python_numpy}
```

MechanicsDSL reports: *Energy conserved (Noether). Normal modes: $\omega_1=\sqrt{g/l}$, $\omega_2=\sqrt{g/l+2k/m}$*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

In [ ]:
m, l, k, g = 1.0, 1.0, 0.5, 9.81
omega1 = np.sqrt(g/l)
omega2 = np.sqrt(g/l + 2*k/m)
omega_beat = (omega2 - omega1) / 2
T_beat = 2*np.pi / omega_beat
print(f'omega1 = {omega1:.4f} rad/s')
print(f'omega2 = {omega2:.4f} rad/s')
print(f'T_beat = {T_beat:.3f} s')

In [ ]:
def eom(t, y, m=m, l=l, k=k, g=g):
    th1, th2, w1, w2 = y
    return [w1, w2, -(g/l)*np.sin(th1)-(k/m)*(th1-th2), -(g/l)*np.sin(th2)+(k/m)*(th1-th2)]

def energy(y, m=m, l=l, k=k, g=g):
    th1, th2, w1, w2 = y
    return 0.5*m*l**2*(w1**2+w2**2) - m*g*l*(np.cos(th1)+np.cos(th2)) + 0.5*k*l**2*(th1-th2)**2

def simulate(th1_0, th2_0, t_end=60.0):
    t = np.arange(0, t_end, 0.005)
    return solve_ivp(eom, [0,t_end], [th1_0,th2_0,0,0], t_eval=t, method='DOP853', rtol=1e-10, atol=1e-12)

## Normal Modes
**Symmetric** ($\theta_1=\theta_2$): spring unstretched, $\omega_1$. **Antisymmetric** ($\theta_1=-\theta_2$): spring maximally stretched, $\omega_2$.

In [ ]:
A = 0.3
sol_sym  = simulate(A, A, 20)
sol_anti = simulate(A, -A, 20)
fig, axes = plt.subplots(2, 1, figsize=(11,6))
fig.suptitle('Normal Modes', fontweight='bold')
axes[0].plot(sol_sym.t, sol_sym.y[0], label=r'$\theta_1$')
axes[0].plot(sol_sym.t, sol_sym.y[1], '--', label=r'$\theta_2$')
axes[0].set_title(fr'Symmetric mode $\omega_1={omega1:.3f}$ rad/s'); axes[0].legend()
axes[1].plot(sol_anti.t, sol_anti.y[0], label=r'$\theta_1$')
axes[1].plot(sol_anti.t, sol_anti.y[1], '--', label=r'$\theta_2$')
axes[1].set_title(fr'Antisymmetric mode $\omega_2={omega2:.3f}$ rad/s'); axes[1].legend()
plt.tight_layout(); plt.show()

## Beating
With $\theta_1^0=A$, $\theta_2^0=0$, energy oscillates completely between the two pendulums with period $T_{beat}=2\pi/(\omega_2-\omega_1)$.

In [ ]:
sol = simulate(A, 0.0, 3*T_beat)
E = np.array([energy(sol.y[:,i]) for i in range(sol.y.shape[1])])
E_drift = np.abs((E-E[0])/E[0])
E1 = 0.5*m*l**2*sol.y[2]**2 - m*g*l*np.cos(sol.y[0])
E2 = 0.5*m*l**2*sol.y[3]**2 - m*g*l*np.cos(sol.y[1])
fig, axes = plt.subplots(3,1,figsize=(11,9))
fig.suptitle(fr'Beating ($T_{{beat}}={T_beat:.2f}$ s)', fontweight='bold')
axes[0].plot(sol.t, sol.y[0], label=r'$\theta_1$'); axes[0].plot(sol.t, sol.y[1], label=r'$\theta_2$')
axes[0].set_ylabel('Angle (rad)'); axes[0].legend()
axes[1].plot(sol.t, E1-E1[0], label='Pendulum 1'); axes[1].plot(sol.t, E2-E2[0], label='Pendulum 2')
axes[1].set_ylabel('Energy transfer (J)'); axes[1].legend()
axes[2].semilogy(sol.t, E_drift, color='tab:red')
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel(r'$|\Delta E/E_0|$')
axes[2].set_title('Energy conservation (Noether)')
plt.tight_layout(); plt.show()
print(f'Max energy drift: {E_drift.max():.2e}')

## Modal Decomposition
$q_+ = (\theta_1+\theta_2)/\sqrt{2}$ oscillates at $\omega_1$; $q_- = (\theta_1-\theta_2)/\sqrt{2}$ at $\omega_2$. FFT recovers both frequencies exactly.

In [ ]:
q_plus  = (sol.y[0]+sol.y[1])/np.sqrt(2)
q_minus = (sol.y[0]-sol.y[1])/np.sqrt(2)
dt = sol.t[1]-sol.t[0]
freqs = np.fft.rfftfreq(len(q_plus), dt)*2*np.pi
peak_plus  = freqs[np.argmax(np.abs(np.fft.rfft(q_plus)))]
peak_minus = freqs[np.argmax(np.abs(np.fft.rfft(q_minus)))]
fig, axes = plt.subplots(2,1,figsize=(11,6))
axes[0].plot(sol.t, q_plus, label=r'$q_+$ numerical')
axes[0].plot(sol.t, (A/np.sqrt(2))*np.cos(omega1*sol.t), '--', alpha=0.7, label=fr'$q_+$ analytical $\omega_1={omega1:.3f}$')
axes[0].set_ylabel('Mode amplitude'); axes[0].legend()
axes[1].plot(sol.t, q_minus, color='tab:orange', label=r'$q_-$ numerical')
axes[1].plot(sol.t, (A/np.sqrt(2))*np.cos(omega2*sol.t), '--', alpha=0.7, color='tab:orange', label=fr'$q_-$ analytical $\omega_2={omega2:.3f}$')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Mode amplitude'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f'FFT: q+ peak = {peak_plus:.4f} rad/s (analytical {omega1:.4f})')
print(f'FFT: q- peak = {peak_minus:.4f} rad/s (analytical {omega2:.4f})')

## Summary
| Property | Value |
|----------|-------|
| Symmetric mode | $\omega_1=\sqrt{g/l}$ |
| Antisymmetric mode | $\omega_2=\sqrt{g/l+2k/m}$ |
| Beat period | $T=2\pi/(\omega_2-\omega_1)$ |
| Conserved quantity | Total energy $H$ (Noether) |

**Next:** [03 — Constraints: Lagrange Multipliers and Baumgarte Stabilization](03_constraints.ipynb)